In [ ]:
#데이터 및 모델, 라이브러리
import pandas as pd
import joblib

df = pd.read_csv("cve_preprocessed.csv")
model = joblib.load("risk_model.pkl")

In [ ]:
#모델 예측
model_features = [
    "attack_vector",
    "attack_complexity",
    "privileges_required",
    "user_interaction",
    "cwe",
    "description",
]

df["predicted_severity"] = model.predict(
    df[model_features]
)

In [ ]:
#점수표
severity_score_map = {
    "LOW": 25,
    "MEDIUM": 50,
    "HIGH": 75,
    "CRITICAL": 100,
}

attack_vector_score_map = {
    "PHYSICAL": 20,
    "LOCAL": 40,
    "ADJACENT_NETWORK": 70,
    "NETWORK": 100,
}

attack_complexity_score_map = {
    "HIGH": 40,
    "LOW": 100,
}

privileges_score_map = {
    "HIGH": 30,
    "LOW": 70,
    "NONE": 100,
}

user_interaction_score_map = {
    "REQUIRED": 40,
    "NONE": 100,
}

In [ ]:
#우선순위 함수
def calculate_priority_score(row) -> float:
    cvss_score = float(row["cvss_score"]) * 10

    severity_score = severity_score_map.get(
        row["predicted_severity"],
        0,
    )

    vector_score = attack_vector_score_map.get(
        row["attack_vector"],
        0,
    )

    complexity_score = attack_complexity_score_map.get(
        row["attack_complexity"],
        0,
    )

    privilege_score = privileges_score_map.get(
        row["privileges_required"],
        0,
    )

    interaction_score = user_interaction_score_map.get(
        row["user_interaction"],
        0,
    )

    score = (
        cvss_score * 0.50
        + severity_score * 0.20
        + vector_score * 0.10
        + complexity_score * 0.08
        + privilege_score * 0.07
        + interaction_score * 0.05
    )

    return round(score, 2)

In [ ]:
#점수와 순위 생성
df["priority_score"] = df.apply(
    calculate_priority_score,
    axis=1,
)

df = df.sort_values(
    by="priority_score",
    ascending=False,
).reset_index(drop=True)

df["priority_rank"] = df.index + 1

display(
    df[
        [
            "priority_rank",
            "cve_id",
            "cvss_score",
            "predicted_severity",
            "attack_vector",
            "attack_complexity",
            "privileges_required",
            "user_interaction",
            "priority_score",
        ]
    ].head(20)
)

,priority_rank,cve_id,cvss_score,predicted_severity,attack_vector,attack_complexity,privileges_required,user_interaction,priority_score
0,1,CVE-2026-54350,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0
1,2,CVE-2026-48055,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0
2,3,CVE-2026-56004,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0
3,4,CVE-2026-13782,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0
4,5,CVE-2026-46840,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0
5,6,CVE-2026-10134,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0
6,7,CVE-2026-54763,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0
7,8,CVE-2026-40772,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0
8,9,CVE-2026-57572,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0
9,10,CVE-2026-52704,10.0,CRITICAL,NETWORK,LOW,NONE,NONE,100.0


In [ ]:
#csv 생성
df.to_csv(
    "priority.csv",
    index=False,
    encoding="utf-8-sig"
)

print("priority.csv 생성 완료")
print("결과 크기:", df.shape)

priority.csv 생성 완료
결과 크기: (9994, 12)
